In [1]:
import numpy as np
from config import (SPOT, STRIKE, MATURITY, VOL, MU, DT, KAPPA, C, INIT_POSITION, R, TAU, GAMMA, LEARN_RATE, STATE_DIM, ACTION_DIM, HIDDEN_DIM, BATCH_SIZE,
                    ACTIONS_LIST, ACTION_DIMENSION, EPISODES, SCORE_WINDOW_LENGTH, STOP_AVG_REWARD, PLOT, REPORT)

from env.hedging_env import HedgingEnv
from utils.bs import bs_delta, bs_price
from utils.compute_cost import compute_cost
from utils.generate_report import build_report
from utils.print import (print_hedge_table, plot_histogram, plot_cost_bars, plot_learningcurve, plot_learningcurve_grid, 
                         plot_policy_heatmaps, plot_hedge_trajectory, plot_hybrid_decomposition, get_trajectory_data, plot_policy_3d, get_policy_3d_grids)
from utils.policy import (make_policy_BSM, make_policy_DDPG, make_policy_TD3, make_policy_DQN, make_policy_Hybrid)

from train.train_DDPG_TD3 import train_DDPG_TD3, train_DDPG_TD3_without_OU_noise
from train.train_DQN import train_DQN
from train.train_hybrid import train_hybrid, train_hybrid_sequential

from models.dqn_agent import DQNAgent
from models.td3_agent import TD3Agent
from models.ddpg_agent import DDPGAgent
from models.hybrid_agent import HybridAgent
import matplotlib as plt
import os as os
import torch 
import pickle

In [2]:
env = HedgingEnv(SPOT, STRIKE, MATURITY, VOL, MU, DT, KAPPA, C, INIT_POSITION, R)
# Initiera agenter med samma config som vid träning
dqn_eval = DQNAgent(STATE_DIM, ACTION_DIMENSION, HIDDEN_DIM, TAU, GAMMA, LEARN_RATE)
td3_eval = TD3Agent(STATE_DIM, ACTION_DIM, HIDDEN_DIM, TAU, GAMMA, LEARN_RATE)
ddpg_eval = DDPGAgent(STATE_DIM, ACTION_DIM, HIDDEN_DIM, TAU, GAMMA, LEARN_RATE)

hybrid_dqn_eval = DQNAgent(STATE_DIM, ACTION_DIMENSION, HIDDEN_DIM, TAU, GAMMA, LEARN_RATE)
hybrid_td3_eval = TD3Agent(STATE_DIM, ACTION_DIM, HIDDEN_DIM, TAU, GAMMA, LEARN_RATE)
hybrid_agent_eval = HybridAgent(hybrid_dqn_eval, hybrid_td3_eval, ACTIONS_LIST)

# Ladda de sparade vikterna
dqn_eval.qnet.load_state_dict(torch.load("../saved_models/dqn_qnet.pth"))
td3_eval.actor.load_state_dict(torch.load("../saved_models/td3_actor.pth"))
ddpg_eval.actor.load_state_dict(torch.load("../saved_models/ddpg_actor.pth"))

hybrid_dqn_eval.qnet.load_state_dict(torch.load("../saved_models/hybrid_dqn.pth"))
hybrid_td3_eval.actor.load_state_dict(torch.load("../saved_models/hybrid_td3.pth"))

# Sätt i evaluation mode (viktigt för t.ex. Batch Normalization/Dropout)

dqn_eval.qnet.eval()
td3_eval.actor.eval()
ddpg_eval.actor.eval()

hybrid_dqn_eval.qnet.eval()
hybrid_td3_eval.actor.eval()

print("Hjärnorna är laddade och redo för testning!")

Hjärnorna är laddade och redo för testning!


In [3]:

#### Report and Plots ###
with open('../saved_models/cost_history.pkl', 'rb') as f:
    cost_data = pickle.load(f)

Cost_DQN = cost_data["Cost DQN"]
Cost_DDPG = cost_data["Cost DDPG"]
Cost_TD3 = cost_data["Cost TD3"]
Cost_HYBRID = cost_data["Cost HYBRID"]
OptionPrice = cost_data["Options Price"]
Cost_BSM = cost_data["Cost BSM"]



In [4]:
#### print hedge table  ###

print_hedge_table(Cost_BSM, Cost_DDPG, Cost_DQN, Cost_TD3, Cost_HYBRID, OptionPrice)


  Hedging Performance Summary
                                     BSM    DDPG      DQN      TD3   Hybrid
Mean hedge cost (% option price)  90.448  43.206   17.330   11.904   12.007
Std hedge cost  (% option price)  35.596  74.618  119.473  158.557  158.937



In [6]:
#### Report and Plots ###
with open('../saved_models/rewards_history.pkl', 'rb') as f:
    episode_rewards = pickle.load(f)

episode_rewards_DQN = episode_rewards["DQN"]
episode_rewards_DDPG = episode_rewards["DDPG"]
episode_rewards_TD3 = episode_rewards["TD3"]
episode_rewards_HYBRID = episode_rewards["HYBRID"]

trajectory_data = get_trajectory_data(env, dqn_eval, ddpg_eval, td3_eval, hybrid_agent_eval, ACTIONS_LIST, VOL)
policy_data = get_policy_3d_grids(dqn_eval, ddpg_eval, td3_eval, hybrid_agent_eval, ACTIONS_LIST, MATURITY, VOL)

if REPORT:
    build_report(
        Cost_BSM, Cost_DDPG, Cost_DQN, Cost_TD3, Cost_HYBRID, OptionPrice,
        episode_rewards_DDPG, episode_rewards_DQN,
        episode_rewards_TD3, episode_rewards_HYBRID, trajectory_data, policy_data, MATURITY,
        output_path="hedging_report.html"
    )

if PLOT:
    #plot_histogram(Cost_BSM, Cost_DDPG, Cost_DQN, Cost_TD3, Cost_HYBRID, OptionPrice)
    plot_cost_bars(Cost_BSM, Cost_DDPG, Cost_DQN, Cost_TD3, Cost_HYBRID, OptionPrice)
    plot_learningcurve(episode_rewards_DDPG, episode_rewards_DQN,
                       episode_rewards_TD3, episode_rewards_HYBRID)
    plot_learningcurve_grid(episode_rewards_DDPG, episode_rewards_DQN,
                            episode_rewards_TD3, episode_rewards_HYBRID,)
    plot_policy_heatmaps(ddpg_eval, dqn_eval, td3_eval, hybrid_agent_eval, ACTIONS_LIST, MATURITY, VOL),
    plot_hedge_trajectory(env, ddpg_eval, dqn_eval, td3_eval, hybrid_agent_eval, ACTIONS_LIST, VOL)
    plot_hybrid_decomposition(hybrid_agent_eval, ACTIONS_LIST, MATURITY)
    plot_policy_3d(ddpg_eval, dqn_eval, td3_eval, hybrid_agent_eval,
                   ACTIONS_LIST, MATURITY, VOL, n_grid=40)

print("Done!")



TypeError: build_report() got multiple values for argument 'output_path'